#  ARPU (Average Revenue Per User)

### ARPU = Total Revenue / Unique Customer

#### - It tells you how much revenue you earn per customer on average.

In [4]:
import pandas as pd

# Load datasets
orders = pd.read_csv("../Source data/OListDatasets/olist_orders_dataset.csv")
order_items = pd.read_csv("../Source data/OListDatasets/olist_order_items_dataset.csv")

# Inspect columns
print("Orders columns:", orders.columns)
print("Order_items columns:", order_items.columns)

Orders columns: Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Order_items columns: Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')


In [5]:
order_data = order_items.merge(orders[['order_id','customer_id']], on='order_id', how='left')
order_data.head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69,816cbea969fe5b689b39cfc97a506742
6,00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85,32e2e6ab09e778d99bf2e0ecd4898718
7,000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75,9ed5e522dd9dd85b4af4a077526d8117
8,0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65,16150771dfd4776261284213b89c304e
9,0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40,351d3cb2cee3c7fd0af6616c82df21d3


In [6]:
customer_revenue = (
    order_data.groupby('customer_id')['price']
    .sum()
    .reset_index()
    .rename(columns={'price':'customer_revenue'})
)

print(customer_revenue.head(10))

                        customer_id  customer_revenue
0  00012a2ce6f8dcda20d059ce98491703             89.80
1  000161a058600d5901f007fab4c27140             54.90
2  0001fd6190edaaf884bcaf3d49edf079            179.99
3  0002414f95344307404f0ace7a26f1d5            149.90
4  000379cdec625522490c315e70c7a9fb             93.00
5  0004164d20a9e969af783496f3408652             59.99
6  000419c5494106c306a97b5635748086             34.30
7  00046a560d407e99b969756e0b10f282            120.90
8  00050bf6e01e69d5c0fd612f1bcfb69c             69.99
9  000598caf2ef4117407665ac33275130           1107.00


In [7]:
# Calulate ARPU
total_revenue = customer_revenue['customer_revenue'].sum()
unique_customers = customer_revenue['customer_id'].nunique()

arpu = total_revenue / unique_customers
print("Total Revenue:", total_revenue)
print("Unique Customers:", unique_customers)
print("ARPU (Average Revenue Per User):", arpu)

Total Revenue: 13591643.7
Unique Customers: 98666
ARPU (Average Revenue Per User): 137.75407637889444


### # (optional) ARPU over time

In [8]:
# Convert purchase timestamp to datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Merge with order_items
order_data = order_items.merge(orders[['order_id','customer_id','order_purchase_timestamp']], on='order_id', how='left')

# Extract month
order_data['month'] = order_data['order_purchase_timestamp'].dt.to_period('M')

# Calculate ARPU per month
monthly_arpu = (
    order_data.groupby('month')
    .apply(lambda x: x['price'].sum() / x['customer_id'].nunique())
    .reset_index(name='ARPU')
)

print(monthly_arpu.head())

     month        ARPU
0  2016-09   89.120000
1  2016-10  160.739156
2  2016-12   10.900000
3  2017-01  152.487795
4  2017-02  142.702262


C:\Users\ZYadmin\AppData\Local\Temp\ipykernel_1608\1644793956.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x['price'].sum() / x['customer_id'].nunique())
